# Data Warehouse — Modelado Dimensional en Python

## Unidad 4: Infraestructura de Datos

Este notebook implementa un Data Warehouse completo usando Python y SQLite: creamos un esquema estrella desde cero, cargamos datos con un pipeline ETL, ejecutamos consultas analiticas y construimos una dimension de tipo SCD-2 con historial de cambios.

### Contenido:
1. Crear el esquema estrella (hechos + dimensiones)
2. Pipeline ETL: extraer, transformar, cargar
3. Consultas analiticas (OLAP)
4. Slowly Changing Dimensions (SCD Tipo 2)
5. Data Mart: crear una vista para un area de negocio

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime, timedelta
import os

---
## 1. Crear el esquema estrella

Usamos SQLite como motor porque no requiere instalacion — viene con Python. Los conceptos aplican igual a Snowflake, BigQuery o PostgreSQL.

In [ ]:
# ============================================================
# CREAR LA BASE DE DATOS Y LAS TABLAS
# ============================================================

# SQLite guarda todo en un archivo local
db_path = 'warehouse.db'
if os.path.exists(db_path):
    os.remove(db_path)

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print(f"Base de datos creada: {db_path}")

In [ ]:
# ============================================================
# DIMENSION: FECHA
# ============================================================

# La dimension fecha se precalcula con todos los atributos
# que los reportes van a necesitar: dia, mes, trimestre,
# es festivo, es fin de semana, etc.

cursor.execute("""
CREATE TABLE dim_fecha (
    fecha_id     INTEGER PRIMARY KEY,   -- Formato YYYYMMDD
    fecha        DATE NOT NULL,
    dia          INTEGER,
    mes          INTEGER,
    mes_nombre   TEXT,
    trimestre    TEXT,
    anio         INTEGER,
    dia_semana   TEXT,
    es_fin_semana INTEGER,              -- 0 o 1
    semana_anio  INTEGER
)
""")

# Generar 2 anios de fechas
fechas = pd.date_range('2023-01-01', '2024-12-31', freq='D')

registros_fecha = []
for f in fechas:
    registros_fecha.append((
        int(f.strftime('%Y%m%d')),        # fecha_id: 20230101
        f.strftime('%Y-%m-%d'),            # fecha
        f.day,                              # dia
        f.month,                            # mes
        f.strftime('%B'),                   # mes_nombre
        f'Q{f.quarter}',                   # trimestre
        f.year,                             # anio
        f.strftime('%A'),                   # dia_semana
        1 if f.weekday() >= 5 else 0,      # es_fin_semana
        f.isocalendar()[1],                 # semana_anio
    ))

cursor.executemany(
    "INSERT INTO dim_fecha VALUES (?,?,?,?,?,?,?,?,?,?)",
    registros_fecha
)
conn.commit()

print(f"dim_fecha: {len(registros_fecha)} registros cargados")
pd.read_sql("SELECT * FROM dim_fecha LIMIT 5", conn)

In [ ]:
# ============================================================
# DIMENSION: PRODUCTO
# ============================================================

cursor.execute("""
CREATE TABLE dim_producto (
    producto_id   INTEGER PRIMARY KEY,
    nombre        TEXT NOT NULL,
    categoria     TEXT NOT NULL,
    subcategoria  TEXT,
    precio_lista  REAL
)
""")

productos = [
    (101, 'Dashboard Ejecutivo', 'BI', 'Visualizacion', 450.0),
    (102, 'Reporte Automatico', 'BI', 'Reporteria', 280.0),
    (103, 'API REST', 'Integracion', 'Datos', 620.0),
    (104, 'App Web Analitica', 'BI', 'Aplicacion', 350.0),
    (105, 'Pipeline ETL', 'Integracion', 'Infraestructura', 500.0),
    (106, 'Modelo Predictivo', 'ML', 'Modelos', 800.0),
    (107, 'Chatbot Datos', 'ML', 'Aplicacion', 550.0),
]

cursor.executemany(
    "INSERT INTO dim_producto VALUES (?,?,?,?,?)",
    productos
)
conn.commit()

print(f"dim_producto: {len(productos)} registros")
pd.read_sql("SELECT * FROM dim_producto", conn)

In [ ]:
# ============================================================
# DIMENSION: REGION
# ============================================================

cursor.execute("""
CREATE TABLE dim_region (
    region_id     INTEGER PRIMARY KEY,
    ciudad        TEXT NOT NULL,
    departamento  TEXT NOT NULL,
    zona          TEXT NOT NULL
)
""")

regiones = [
    (1, 'Bogota', 'Cundinamarca', 'Centro'),
    (2, 'Medellin', 'Antioquia', 'Noroccidente'),
    (3, 'Cali', 'Valle del Cauca', 'Suroccidente'),
    (4, 'Manizales', 'Caldas', 'Eje Cafetero'),
    (5, 'Barranquilla', 'Atlantico', 'Costa'),
]

cursor.executemany(
    "INSERT INTO dim_region VALUES (?,?,?,?)",
    regiones
)
conn.commit()

print(f"dim_region: {len(regiones)} registros")
pd.read_sql("SELECT * FROM dim_region", conn)

In [ ]:
# ============================================================
# DIMENSION: CLIENTE
# ============================================================

cursor.execute("""
CREATE TABLE dim_cliente (
    cliente_id   INTEGER PRIMARY KEY,
    nombre       TEXT NOT NULL,
    segmento     TEXT NOT NULL,
    canal        TEXT NOT NULL,
    fecha_registro DATE
)
""")

np.random.seed(42)
nombres = ['TechCorp', 'DataSoft', 'Analitika', 'InfoSystems', 'CloudBI',
           'MetricaPro', 'VisionData', 'SmartBI', 'DataDriven', 'AIFactory',
           'NeoAnalytics', 'ByteInsights', 'PredictCo', 'FlowData', 'CoreBI']
segmentos = ['Enterprise', 'Profesional', 'Startup']
canales = ['Directo', 'Referido', 'Organico', 'Evento']

clientes = []
for i, nombre in enumerate(nombres, start=1):
    fecha = (datetime(2022, 1, 1) + timedelta(days=np.random.randint(0, 700))).strftime('%Y-%m-%d')
    clientes.append((
        i, nombre,
        np.random.choice(segmentos, p=[0.2, 0.5, 0.3]),
        np.random.choice(canales),
        fecha
    ))

cursor.executemany(
    "INSERT INTO dim_cliente VALUES (?,?,?,?,?)",
    clientes
)
conn.commit()

print(f"dim_cliente: {len(clientes)} registros")
pd.read_sql("SELECT * FROM dim_cliente", conn)

In [ ]:
# ============================================================
# TABLA DE HECHOS: VENTAS
# ============================================================

cursor.execute("""
CREATE TABLE hechos_ventas (
    venta_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    fecha_id      INTEGER NOT NULL REFERENCES dim_fecha(fecha_id),
    producto_id   INTEGER NOT NULL REFERENCES dim_producto(producto_id),
    region_id     INTEGER NOT NULL REFERENCES dim_region(region_id),
    cliente_id    INTEGER NOT NULL REFERENCES dim_cliente(cliente_id),
    unidades      INTEGER NOT NULL,
    precio_venta  REAL NOT NULL,
    costo         REAL NOT NULL,
    descuento     REAL DEFAULT 0
)
""")

# Generar transacciones sinteticas
np.random.seed(42)
n_ventas = 5000

producto_ids = [p[0] for p in productos]
producto_precios = {p[0]: p[4] for p in productos}
region_ids = [r[0] for r in regiones]
cliente_ids = [c[0] for c in clientes]

ventas = []
for _ in range(n_ventas):
    # Fecha aleatoria en 2023-2024
    fecha = fechas[np.random.randint(0, len(fechas))]
    fecha_id = int(fecha.strftime('%Y%m%d'))
    
    prod_id = np.random.choice(producto_ids)
    precio_lista = producto_precios[prod_id]
    
    # Bogota tiene mas ventas (40%)
    reg_id = np.random.choice(region_ids, p=[0.4, 0.25, 0.15, 0.1, 0.1])
    cli_id = np.random.choice(cliente_ids)
    
    unidades = np.random.randint(1, 25)
    descuento = np.random.choice([0, 0, 0, 5, 10, 15, 20], p=[0.4, 0.15, 0.1, 0.1, 0.1, 0.1, 0.05])
    precio_venta = round(precio_lista * (1 - descuento / 100), 2)
    costo = round(precio_lista * 0.4, 2)
    
    ventas.append((fecha_id, prod_id, reg_id, cli_id, unidades, precio_venta, costo, descuento))

cursor.executemany(
    "INSERT INTO hechos_ventas (fecha_id, producto_id, region_id, cliente_id, unidades, precio_venta, costo, descuento) VALUES (?,?,?,?,?,?,?,?)",
    ventas
)
conn.commit()

print(f"hechos_ventas: {n_ventas} transacciones generadas")
pd.read_sql("SELECT * FROM hechos_ventas LIMIT 5", conn)

In [ ]:
# ============================================================
# VERIFICAR EL ESQUEMA COMPLETO
# ============================================================

tablas = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
    conn
)

print("Tablas en el Warehouse:\n")
for _, row in tablas.iterrows():
    nombre = row['name']
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {nombre}", conn)['n'].iloc[0]
    tipo = 'HECHOS' if nombre.startswith('hechos') else 'DIMENSION'
    print(f"  [{tipo:9s}] {nombre:20s} {count:>6,} registros")

---
## 2. Pipeline ETL

Simulamos la llegada de datos nuevos desde un sistema transaccional (un CSV) y los cargamos al Warehouse pasando por las 3 fases: Extract, Transform, Load.

In [ ]:
# ============================================================
# EXTRACT: simular datos crudos desde el sistema operacional
# ============================================================

# En produccion esto seria un query a la base de datos
# transaccional, o un archivo que deja el ERP

datos_crudos = pd.DataFrame({
    'fecha': ['2024-12-28', '2024-12-28', '2024-12-29',
              '2024-12-29', '2024-12-30', '28/12/2024'],  # Formato inconsistente
    'producto': ['Dashboard Ejecutivo', 'API REST', 'Reporte Automatico',
                 'DASHBOARD EJECUTIVO', None, 'Pipeline ETL'],  # Duplicado en mayusculas + nulo
    'ciudad': ['Bogota', 'Medellin', 'Bogotá', 'bogota',
               'Cali', 'Manizales'],  # Variantes de Bogota
    'cliente': ['TechCorp', 'DataSoft', 'Analitika',
                'TechCorp', 'InfoSystems', 'CloudBI'],
    'unidades': [10, 5, 15, 8, -3, 12],  # Negativo
    'precio': [450.0, 620.0, 280.0, 450.0, 350.0, 500.0],
    'descuento_pct': ['10%', '0', '5%', '15%', '0', '20%'],  # Formato inconsistente
})

print("Datos crudos del sistema operacional:")
datos_crudos

In [ ]:
# ============================================================
# TRANSFORM: limpiar y mapear a las dimensiones del Warehouse
# ============================================================

def transformar_datos(df_crudo, conn):
    """
    Transforma datos crudos del sistema operacional
    al formato del esquema estrella del Warehouse.
    Retorna datos listos para cargar y registros rechazados.
    """
    df = df_crudo.copy()
    rechazados = []
    
    # 1. Limpiar fechas (multiples formatos)
    df['fecha'] = pd.to_datetime(df['fecha'], format='mixed', dayfirst=True, errors='coerce')
    
    # 2. Limpiar producto
    df['producto'] = df['producto'].str.strip().str.title()
    
    # 3. Limpiar ciudad (tildes, case)
    mapeo_ciudad = {'bogotá': 'Bogota', 'bogota': 'Bogota'}
    df['ciudad'] = df['ciudad'].str.strip().str.lower().replace(mapeo_ciudad).str.title()
    
    # 4. Limpiar descuento
    df['descuento_pct'] = df['descuento_pct'].str.replace('%', '', regex=False)
    df['descuento_pct'] = pd.to_numeric(df['descuento_pct'], errors='coerce').fillna(0)
    
    # 5. Rechazar filas invalidas
    mask_nulo = df['producto'].isna() | df['fecha'].isna()
    mask_negativo = df['unidades'] <= 0
    mask_invalido = mask_nulo | mask_negativo
    
    if mask_invalido.any():
        rechazados = df[mask_invalido].copy()
        rechazados['motivo'] = np.where(mask_nulo, 'valor_nulo', 'unidades_negativas')
        df = df[~mask_invalido].copy()
    
    # 6. Mapear a claves del Warehouse
    df['fecha_id'] = df['fecha'].dt.strftime('%Y%m%d').astype(int)
    
    # Buscar producto_id
    dim_prod = pd.read_sql("SELECT producto_id, nombre FROM dim_producto", conn)
    df = df.merge(dim_prod, left_on='producto', right_on='nombre', how='left')
    
    # Buscar region_id
    dim_reg = pd.read_sql("SELECT region_id, ciudad FROM dim_region", conn)
    df = df.merge(dim_reg, on='ciudad', how='left')
    
    # Buscar cliente_id
    dim_cli = pd.read_sql("SELECT cliente_id, nombre FROM dim_cliente", conn)
    df = df.merge(dim_cli, left_on='cliente', right_on='nombre', how='left', suffixes=('', '_cli'))
    
    # Calcular metricas
    df['precio_venta'] = df['precio'] * (1 - df['descuento_pct'] / 100)
    df['costo'] = df['precio'] * 0.4
    
    # Seleccionar columnas del esquema
    resultado = df[['fecha_id', 'producto_id', 'region_id', 'cliente_id',
                    'unidades', 'precio_venta', 'costo', 'descuento_pct']].copy()
    resultado = resultado.rename(columns={'descuento_pct': 'descuento'})
    resultado = resultado.dropna()  # Quitar los que no mapearon
    
    return resultado, pd.DataFrame(rechazados) if len(rechazados) > 0 else pd.DataFrame()

datos_transformados, datos_rechazados = transformar_datos(datos_crudos, conn)

print(f"Transformados: {len(datos_transformados)} filas listas para cargar")
print(f"Rechazados: {len(datos_rechazados)} filas")

if len(datos_rechazados) > 0:
    print("\nRegistros rechazados:")
    print(datos_rechazados[['fecha', 'producto', 'unidades', 'motivo']].to_string(index=False))

print("\nDatos listos para cargar:")
datos_transformados

In [ ]:
# ============================================================
# LOAD: cargar al Warehouse
# ============================================================

antes = pd.read_sql("SELECT COUNT(*) as n FROM hechos_ventas", conn)['n'].iloc[0]

# Insertar los datos transformados
datos_transformados.to_sql('hechos_ventas', conn, if_exists='append', index=False)

despues = pd.read_sql("SELECT COUNT(*) as n FROM hechos_ventas", conn)['n'].iloc[0]

print(f"Carga completada:")
print(f"  Antes: {antes:,} filas")
print(f"  Cargadas: {despues - antes} filas")
print(f"  Despues: {despues:,} filas")

---
## 3. Consultas analiticas (OLAP)

Las consultas sobre el Warehouse usan JOINs entre la tabla de hechos y las dimensiones para responder preguntas de negocio.

In [ ]:
# ============================================================
# CONSULTA 1: Ingreso por region y trimestre
# ============================================================

query_1 = """
SELECT
    r.ciudad,
    r.zona,
    f.trimestre,
    f.anio,
    SUM(v.unidades * v.precio_venta) AS ingreso,
    SUM(v.unidades * v.costo) AS costo_total,
    SUM(v.unidades * v.precio_venta) - SUM(v.unidades * v.costo) AS margen,
    COUNT(*) AS transacciones
FROM hechos_ventas v
JOIN dim_region r ON v.region_id = r.region_id
JOIN dim_fecha f ON v.fecha_id = f.fecha_id
WHERE f.anio = 2024
GROUP BY r.ciudad, r.zona, f.trimestre, f.anio
ORDER BY r.ciudad, f.trimestre
"""

resultado_1 = pd.read_sql(query_1, conn)
print("Ingreso por region y trimestre (2024):")
resultado_1

In [ ]:
# ============================================================
# CONSULTA 2: Top productos por ingreso
# ============================================================

query_2 = """
SELECT
    p.nombre AS producto,
    p.categoria,
    COUNT(*) AS transacciones,
    SUM(v.unidades) AS unidades_totales,
    ROUND(SUM(v.unidades * v.precio_venta), 0) AS ingreso_total,
    ROUND(AVG(v.descuento), 1) AS descuento_promedio
FROM hechos_ventas v
JOIN dim_producto p ON v.producto_id = p.producto_id
GROUP BY p.nombre, p.categoria
ORDER BY ingreso_total DESC
"""

resultado_2 = pd.read_sql(query_2, conn)
print("Top productos por ingreso:")
resultado_2

In [ ]:
# ============================================================
# CONSULTA 3: Ingreso mensual con tendencia
# ============================================================

query_3 = """
SELECT
    f.anio,
    f.mes,
    f.mes_nombre,
    COUNT(DISTINCT v.cliente_id) AS clientes_unicos,
    ROUND(SUM(v.unidades * v.precio_venta), 0) AS ingreso
FROM hechos_ventas v
JOIN dim_fecha f ON v.fecha_id = f.fecha_id
GROUP BY f.anio, f.mes, f.mes_nombre
ORDER BY f.anio, f.mes
"""

resultado_3 = pd.read_sql(query_3, conn)
print("Ingreso mensual:")
resultado_3

In [ ]:
# ============================================================
# CONSULTA 4: Analisis por segmento de cliente
# ============================================================

query_4 = """
SELECT
    c.segmento,
    COUNT(DISTINCT c.cliente_id) AS clientes,
    COUNT(*) AS transacciones,
    ROUND(SUM(v.unidades * v.precio_venta), 0) AS ingreso_total,
    ROUND(SUM(v.unidades * v.precio_venta) * 1.0 / COUNT(DISTINCT c.cliente_id), 0) AS ingreso_por_cliente,
    ROUND(AVG(v.unidades), 1) AS unidades_promedio
FROM hechos_ventas v
JOIN dim_cliente c ON v.cliente_id = c.cliente_id
GROUP BY c.segmento
ORDER BY ingreso_total DESC
"""

resultado_4 = pd.read_sql(query_4, conn)
print("Analisis por segmento de cliente:")
resultado_4

In [ ]:
# ============================================================
# CONSULTA 5: Fin de semana vs entre semana
# ============================================================

# Esto demuestra el valor de la dimension fecha precalculada:
# no necesitas calcular si es fin de semana en el query

query_5 = """
SELECT
    CASE WHEN f.es_fin_semana = 1 THEN 'Fin de semana' ELSE 'Entre semana' END AS periodo,
    COUNT(*) AS transacciones,
    ROUND(SUM(v.unidades * v.precio_venta), 0) AS ingreso,
    ROUND(AVG(v.unidades * v.precio_venta), 0) AS ticket_promedio
FROM hechos_ventas v
JOIN dim_fecha f ON v.fecha_id = f.fecha_id
GROUP BY f.es_fin_semana
"""

resultado_5 = pd.read_sql(query_5, conn)
print("Ventas: fin de semana vs entre semana:")
resultado_5

---
## 4. Slowly Changing Dimensions (SCD Tipo 2)

Las dimensiones cambian. Un cliente cambia de segmento, una ciudad cambia de zona, un producto cambia de precio. El SCD Tipo 2 conserva el historial creando una nueva fila por cada cambio.

In [ ]:
# ============================================================
# CREAR DIMENSION CLIENTE CON SCD TIPO 2
# ============================================================

cursor.execute("""
CREATE TABLE dim_cliente_scd2 (
    sk_cliente    INTEGER PRIMARY KEY AUTOINCREMENT,  -- Surrogate key
    cliente_id    INTEGER NOT NULL,                   -- Natural key (la del sistema origen)
    nombre        TEXT NOT NULL,
    segmento      TEXT NOT NULL,
    canal         TEXT NOT NULL,
    ciudad        TEXT NOT NULL,
    vigente_desde DATE NOT NULL,
    vigente_hasta DATE NOT NULL,
    es_actual     INTEGER NOT NULL DEFAULT 1          -- 1 = vigente, 0 = historico
)
""")

# Cargar estado inicial
clientes_scd = [
    (1, 'TechCorp', 'Startup', 'Directo', 'Medellin', '2022-01-15', '9999-12-31', 1),
    (2, 'DataSoft', 'Profesional', 'Referido', 'Bogota', '2022-03-01', '9999-12-31', 1),
    (3, 'Analitika', 'Startup', 'Organico', 'Cali', '2022-06-15', '9999-12-31', 1),
]

cursor.executemany(
    "INSERT INTO dim_cliente_scd2 (cliente_id, nombre, segmento, canal, ciudad, vigente_desde, vigente_hasta, es_actual) VALUES (?,?,?,?,?,?,?,?)",
    clientes_scd
)
conn.commit()

print("Estado inicial:")
pd.read_sql("SELECT * FROM dim_cliente_scd2", conn)

In [ ]:
# ============================================================
# APLICAR UN CAMBIO: TechCorp paso de Startup a Enterprise
# y se mudo de Medellin a Bogota
# ============================================================

def aplicar_cambio_scd2(conn, cliente_id, nuevos_datos, fecha_cambio):
    """
    Aplica un cambio SCD Tipo 2:
    1. Cierra el registro actual (vigente_hasta = fecha_cambio - 1)
    2. Crea un registro nuevo con los datos actualizados
    """
    cursor = conn.cursor()
    fecha_cierre = (datetime.strptime(fecha_cambio, '%Y-%m-%d') - timedelta(days=1)).strftime('%Y-%m-%d')
    
    # 1. Cerrar el registro actual
    cursor.execute("""
        UPDATE dim_cliente_scd2
        SET vigente_hasta = ?,
            es_actual = 0
        WHERE cliente_id = ?
          AND es_actual = 1
    """, (fecha_cierre, cliente_id))
    
    # 2. Insertar el registro nuevo
    cursor.execute("""
        INSERT INTO dim_cliente_scd2
        (cliente_id, nombre, segmento, canal, ciudad, vigente_desde, vigente_hasta, es_actual)
        VALUES (?, ?, ?, ?, ?, ?, '9999-12-31', 1)
    """, (
        cliente_id,
        nuevos_datos['nombre'],
        nuevos_datos['segmento'],
        nuevos_datos['canal'],
        nuevos_datos['ciudad'],
        fecha_cambio
    ))
    
    conn.commit()

# TechCorp cambio de Startup a Enterprise y se mudo a Bogota el 2024-07-01
aplicar_cambio_scd2(conn, cliente_id=1, nuevos_datos={
    'nombre': 'TechCorp',
    'segmento': 'Enterprise',
    'canal': 'Directo',
    'ciudad': 'Bogota',
}, fecha_cambio='2024-07-01')

print("Despues del cambio — TechCorp tiene 2 filas:")
pd.read_sql("""
    SELECT * FROM dim_cliente_scd2
    WHERE cliente_id = 1
    ORDER BY vigente_desde
""", conn)

In [ ]:
# ============================================================
# CONSULTAR CON HISTORIAL
# ============================================================

# Consulta que respeta el historial:
# las ventas de TechCorp en 2023 se atribuyen a Medellin/Startup
# las ventas de TechCorp en 2024-Q3+ se atribuyen a Bogota/Enterprise

query_scd = """
SELECT
    c.nombre,
    c.segmento,
    c.ciudad,
    c.vigente_desde,
    c.vigente_hasta,
    c.es_actual
FROM dim_cliente_scd2 c
ORDER BY c.cliente_id, c.vigente_desde
"""

print("Dimension completa con historial:")
pd.read_sql(query_scd, conn)

---
## 5. Data Mart

Un Data Mart es una vista del Warehouse enfocada en un area de negocio. En vez de que cada analista escriba el mismo JOIN de 5 tablas, se crea una vista precalculada.

In [ ]:
# ============================================================
# CREAR UN DATA MART DE VENTAS
# ============================================================

# Una vista que une hechos + dimensiones y precalcula metricas
# El analista de ventas solo hace SELECT sobre esta vista

cursor.execute("""
CREATE VIEW IF NOT EXISTS mart_ventas AS
SELECT
    f.fecha,
    f.anio,
    f.mes_nombre AS mes,
    f.trimestre,
    f.dia_semana,
    p.nombre AS producto,
    p.categoria AS categoria_producto,
    r.ciudad,
    r.zona,
    c.nombre AS cliente,
    c.segmento,
    v.unidades,
    v.precio_venta,
    v.costo,
    v.descuento,
    v.unidades * v.precio_venta AS ingreso,
    v.unidades * v.costo AS costo_total,
    v.unidades * (v.precio_venta - v.costo) AS margen
FROM hechos_ventas v
JOIN dim_fecha f ON v.fecha_id = f.fecha_id
JOIN dim_producto p ON v.producto_id = p.producto_id
JOIN dim_region r ON v.region_id = r.region_id
JOIN dim_cliente c ON v.cliente_id = c.cliente_id
""")
conn.commit()

print("Vista mart_ventas creada.")
print("Ahora el analista solo escribe:")
print("  SELECT ciudad, SUM(ingreso) FROM mart_ventas GROUP BY ciudad")

In [ ]:
# ============================================================
# USAR EL DATA MART — queries simples
# ============================================================

# Ahora las consultas son triviales — sin JOINs

print("=== Ingreso por ciudad (2024) ===")
print(pd.read_sql("""
    SELECT ciudad, ROUND(SUM(ingreso), 0) AS ingreso,
           ROUND(SUM(margen), 0) AS margen
    FROM mart_ventas WHERE anio = 2024
    GROUP BY ciudad ORDER BY ingreso DESC
""", conn).to_string(index=False))

print("\n=== Ingreso por categoria (2024) ===")
print(pd.read_sql("""
    SELECT categoria_producto, ROUND(SUM(ingreso), 0) AS ingreso,
           COUNT(*) AS transacciones
    FROM mart_ventas WHERE anio = 2024
    GROUP BY categoria_producto ORDER BY ingreso DESC
""", conn).to_string(index=False))

print("\n=== Top 5 clientes (2024) ===")
print(pd.read_sql("""
    SELECT cliente, segmento, ROUND(SUM(ingreso), 0) AS ingreso
    FROM mart_ventas WHERE anio = 2024
    GROUP BY cliente, segmento ORDER BY ingreso DESC LIMIT 5
""", conn).to_string(index=False))

In [ ]:
# ============================================================
# EXPORTAR EL DATA MART A PARQUET
# ============================================================

# Para conectar con Power BI o Streamlit,
# se puede exportar la vista como Parquet

df_mart = pd.read_sql("SELECT * FROM mart_ventas", conn)
df_mart.to_parquet('mart_ventas.parquet')

print(f"Data Mart exportado: {len(df_mart):,} filas")
print(f"Archivo: mart_ventas.parquet ({os.path.getsize('mart_ventas.parquet') / 1e6:.1f} MB)")
print(f"\nColumnas: {df_mart.columns.tolist()}")

In [ ]:
# Cerrar la conexion y limpiar
conn.close()
for f in ['warehouse.db', 'mart_ventas.parquet']:
    if os.path.exists(f):
        os.remove(f)
print("Archivos de prueba eliminados.")

---
## Resumen

| Concepto | Lo que importa |
|---|---|
| **Esquema estrella** | Tabla de hechos (metricas) + dimensiones (contexto). Los JOINs conectan todo |
| **dim_fecha** | Precalcular atributos (trimestre, dia semana, festivo) facilita los reportes |
| **ETL** | Extract (traer), Transform (limpiar y mapear), Load (cargar). Los rechazados se registran |
| **SCD Tipo 2** | Conservar historial de cambios con vigente_desde / vigente_hasta |
| **Data Mart** | Vista precalculada para que el analista haga queries simples sin JOINs |
| **Claves** | Natural key (del sistema origen) vs Surrogate key (generada por el Warehouse) |

### Siguiente paso
En el notebook de **Data Lake** veremos como almacenar datos crudos en multiples formatos, organizar zonas Bronze/Silver/Gold, y por que Parquet es el formato dominante.